In [1]:
!pip install -q mygene
!git lfs install
!GIT_LFS_SKIP_SMUDGE=1 git clone https://huggingface.co/ctheodoris/Geneformer /content/Geneformer
!cd /content/Geneformer && pip install -q .
!pip install -q transformers==4.49.0
!python -c "import numpy,numpy.char; from scipy import sparse; from scipy.stats import spearmanr; print('>>> HEALTHY <<<')"
print("=== setup finished ===")

Git LFS initialized.
fatal: destination path '/content/Geneformer' already exists and is not an empty directory.
  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.66.0 which is incompatible.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.5 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.5 which is

In [2]:
import geneformer
from geneformer import TranscriptomeTokenizer, EmbExtractor
import anndata, scanpy, transformers, numpy, torch
print("all imports OK")
print("transformers:", transformers.__version__, "| numpy:", numpy.__version__)
print("GPU:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

all imports OK
transformers: 4.49.0 | numpy: 2.2.6
GPU: True Tesla T4


In [4]:
import os, zipfile, glob, shutil

# 1. Find and unzip processed.zip wherever it mounted
zips = glob.glob("/kaggle/input/**/processed.zip", recursive=True)
if zips:
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall("/kaggle/working/unzipped")
    print("unzipped:", zips[0])
else:
    print("no processed.zip found under /kaggle/input — check the dataset upload")

# 2. Locate the processed dir by a signature file (nesting-proof)
hits = glob.glob("/kaggle/working/**/gene_columns.json", recursive=True) \
     + glob.glob("/kaggle/input/**/gene_columns.json", recursive=True)
if not hits:
    raise SystemExit("gene_columns.json not found — is it inside processed.zip?")
PROCESSED = os.path.dirname(hits[0])
os.environ["DEPMAP_PROCESSED_DIR"] = PROCESSED
print("PROCESSED_DIR =", PROCESSED)
print("contents:", sorted(os.listdir(PROCESSED)))

# 3. Copy the five .py files into the working dir (they mount read-only)
for f in ["config.py","io_utils.py","gene_ids.py",
          "prepare_geneformer_input.py","run_geneformer_embeddings.py"]:
    found = glob.glob(f"/kaggle/input/**/{f}", recursive=True)
    if found:
        shutil.copy(found[0], f"/kaggle/working/{f}")
missing = [f for f in ["config.py","io_utils.py","gene_ids.py",
           "prepare_geneformer_input.py","run_geneformer_embeddings.py"]
           if not os.path.exists(f"/kaggle/working/{f}")]
print("missing .py:", missing if missing else "none — all present")

no processed.zip found under /kaggle/input — check the dataset upload
PROCESSED_DIR = /kaggle/input/datasets/felixxchn/dep-map-processed/processed
contents: ['baseline_results.json', 'crispr_effect.labels.json', 'crispr_effect.npz', 'expression.labels.json', 'expression.npz', 'gene_columns.json', 'gene_id_map.csv', 'join_report.json', 'join_report.txt', 'model_metadata.csv', 'selective_genes.json', 'splits.json']
missing .py: none — all present


In [7]:
cfg_path = "/kaggle/working/config.py"
with open(cfg_path) as f:
    src = f.read()

if '"expression_counts"' in src:
    print("already present — no patch needed")
else:
    anchor = "FILE_ALIASES: dict[str, tuple[list[str], str]] = {"
    block = '''
    "expression_counts": (
        [
            "OmicsExpressionGenesExpectedCountProfile.csv",
            "OmicsExpressionGenesExpectedCount.csv",
        ],
        "OmicsExpression*ExpectedCount*.csv",
    ),'''
    if anchor in src:
        src = src.replace(anchor, anchor + block, 1)
        with open(cfg_path, "w") as f:
            f.write(src)
        print("patched OK — expression_counts alias added")
    else:
        print("ANCHOR NOT FOUND — paste me your config.py and I'll adjust")

patched OK — expression_counts alias added


In [10]:
!cd /kaggle/working && python prepare_geneformer_input.py

PREPARING GENEFORMER INPUT

[1/4] Expression counts
No expected-counts file found -- reconstructing pseudo-counts from log-TPM.
  For a reported result, download DepMap's gene-level expected-counts expression file and re-run; see README.
      source     : reconstructed_from_log_tpm
      cell lines : 1140

[2/4] Ensembl mapping
Querying mygene.info for 18460 Entrez IDs...
Cached Ensembl map to ensembl_map.csv
      mapped   : 18460 / 18460
      unmapped : 0

[3/4] Building AnnData frames
      genes kept        : 18460
      genes dropped     : 0
      cells             : 1140
      median n_counts   : 782190

[4/4] Writing .h5ad
      wrote geneformer_input.h5ad

DONE
  count source : reconstructed_from_log_tpm
  genes        : 18460 (Ensembl-mapped, in canonical order)
  cells        : 1140

  NOTE: pseudo-counts were used. For a reported result, add DepMap's
  expected-counts file and re-run. See README.

  Next: tokenise + extract embeddings on Kaggle (run_geneformer_embeddings.p

In [9]:
import os, shutil, glob

# find the read-only processed dir by its signature file
hits = glob.glob("/kaggle/input/**/gene_columns.json", recursive=True)
src = os.path.dirname(hits[0])

# copy it into the writable working area
dst = "/kaggle/working/processed"
os.makedirs(dst, exist_ok=True)
for f in os.listdir(src):
    s = os.path.join(src, f)
    if os.path.isfile(s):
        shutil.copy(s, os.path.join(dst, f))

os.environ["DEPMAP_PROCESSED_DIR"] = dst      # repoint to the writable copy
print("PROCESSED_DIR (writable) =", dst)
print("contents:", sorted(os.listdir(dst)))

PROCESSED_DIR (writable) = /kaggle/working/processed
contents: ['baseline_results.json', 'crispr_effect.labels.json', 'crispr_effect.npz', 'expression.labels.json', 'expression.npz', 'gene_columns.json', 'gene_id_map.csv', 'join_report.json', 'join_report.txt', 'model_metadata.csv', 'selective_genes.json', 'splits.json']


In [15]:
import anndata as ad, pandas as pd, glob

paths = glob.glob("/kaggle/working/**/geneformer_input.h5ad", recursive=True)
print("found", len(paths), "copy/copies to patch")

for h5ad in paths:
    adata = ad.read_h5ad(h5ad)
    # ensure ensembl_id survives as a COLUMN
    if "ensembl_id" not in adata.var.columns:
        adata.var["ensembl_id"] = adata.var.index.astype(str)
    # give var a plain 0..N index so positional indexing works
    adata.var = adata.var.reset_index(drop=True)
    adata.var_names = [str(i) for i in range(adata.n_vars)]
    adata.write_h5ad(h5ad)
    print("patched:", h5ad)
    print("   var columns :", list(adata.var.columns))
    print("   var index   :", "RangeIndex" if adata.var.index.equals(pd.RangeIndex(adata.n_vars)) else "STILL WRONG")

print("done")

found 2 copy/copies to patch


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


patched: /kaggle/working/processed/geneformer_input.h5ad
   var columns : ['ensembl_id']
   var index   : STILL WRONG


/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


patched: /kaggle/working/processed/geneformer_tokenize_input/geneformer_input.h5ad
   var columns : ['ensembl_id']
   var index   : STILL WRONG
done


In [13]:
import glob, os, shutil, pickle, geneformer
from huggingface_hub import hf_hub_download, list_repo_files

installed = os.path.dirname(geneformer.__file__)

# find every .pkl the installed package expects, and which are broken stubs
def is_stub(path):
    try:
        with open(path, "rb") as f:
            return f.read(20).startswith(b"version https")
    except Exception:
        return True

stub_pkls = [p for p in glob.glob(f"{installed}/**/*.pkl", recursive=True) if is_stub(p)]
print("stub .pkl files needing real data:", len(stub_pkls))
for p in stub_pkls:
    print("   ", os.path.relpath(p, installed))

# map each stub to its path inside the repo and download the real file
repo_files = list_repo_files("ctheodoris/Geneformer")
copied = 0
for stub in stub_pkls:
    name = os.path.basename(stub)
    match = [f for f in repo_files if f.endswith(name)]
    if not match:
        print("no repo match for", name); continue
    real = hf_hub_download("ctheodoris/Geneformer", match[0])
    shutil.copy(real, stub)
    copied += 1
print(f"\ncopied {copied} real dictionaries")

# verify the one the tokenizer opens first
probe = glob.glob(f"{installed}/**/*gene_median*.pkl", recursive=True)
if probe:
    try:
        with open(probe[0], "rb") as f: pickle.load(f)
        print("gene_median loads OK:", os.path.basename(probe[0]))
    except Exception as e:
        print("STILL BROKEN:", e)

stub .pkl files needing real data: 8
    ensembl_mapping_dict_gc104M.pkl
    gene_median_dictionary_gc104M.pkl
    token_dictionary_gc104M.pkl
    gene_name_id_dict_gc104M.pkl
    gene_dictionaries_30m/gene_name_id_dict_gc30M.pkl
    gene_dictionaries_30m/token_dictionary_gc30M.pkl
    gene_dictionaries_30m/ensembl_mapping_dict_gc30M.pkl
    gene_dictionaries_30m/gene_median_dictionary_gc30M.pkl


geneformer/ensembl_mapping_dict_gc104M.p(…):   0%|          | 0.00/3.96M [00:00<?, ?B/s]

geneformer/gene_median_dictionary_gc104M(…):   0%|          | 0.00/1.51M [00:00<?, ?B/s]

geneformer/token_dictionary_gc104M.pkl:   0%|          | 0.00/426k [00:00<?, ?B/s]

geneformer/gene_name_id_dict_gc104M.pkl:   0%|          | 0.00/1.66M [00:00<?, ?B/s]

geneformer/gene_dictionaries_30m/gene_na(…):   0%|          | 0.00/1.12M [00:00<?, ?B/s]

geneformer/gene_dictionaries_30m/token_d(…):   0%|          | 0.00/788k [00:00<?, ?B/s]

geneformer/gene_dictionaries_30m/ensembl(…):   0%|          | 0.00/584k [00:00<?, ?B/s]

geneformer/gene_dictionaries_30m/gene_me(…):   0%|          | 0.00/941k [00:00<?, ?B/s]


copied 8 real dictionaries
gene_median loads OK: gene_median_dictionary_gc104M.pkl


In [16]:
import geneformer, os, re
tok = os.path.join(os.path.dirname(geneformer.__file__), "tokenizer.py")
with open(tok) as f:
    src = f.read()
before = src

# pandas 2.x: Series[int_array] is label-based; force positional with .iloc
src = src.replace('["ensembl_id_collapsed"][coding_miRNA_loc]',
                  '["ensembl_id_collapsed"].iloc[coding_miRNA_loc]')
src = src.replace('["ensembl_id"][coding_miRNA_loc]',
                  '["ensembl_id"].iloc[coding_miRNA_loc]')

if src != before:
    with open(tok, "w") as f:
        f.write(src)
    print("patched:", src.count('.iloc[coding_miRNA_loc]'), ".iloc fixes applied")
else:
    print("pattern not found — paste me the output below")

# show any remaining raw [coding_miRNA_loc] uses, so we catch siblings now
print("\nremaining [coding_miRNA_loc] occurrences:")
for m in re.finditer(r'.{50}\[coding_miRNA_loc\]', src):
    print("   ", m.group(0).strip())

patched: 4 .iloc fixes applied

remaining [coding_miRNA_loc] occurrences:
    for i in adata.var["ensembl_id_collapsed"].iloc[coding_miRNA_loc]
    miRNA_ids = adata.var["ensembl_id_collapsed"].iloc[coding_miRNA_loc]
    for i in data.ra["ensembl_id_collapsed"].iloc[coding_miRNA_loc]
    g_miRNA_ids = data.ra["ensembl_id_collapsed"].iloc[coding_miRNA_loc]


In [21]:
!cd /kaggle/working && python run_geneformer_embeddings.py

GENEFORMER EMBEDDING EXTRACTION

[1/3] Model
Fetching 4 files: 100%|█████████████████████████| 4/4 [00:00<00:00, 1883.81it/s]
  model at: /kaggle/working/geneformer_work/geneformer_repo/Geneformer-V2-104M_CLcancer

[2/3] Tokenise
Tokenizing /kaggle/working/processed/geneformer_tokenize_input/geneformer_input.h5ad
/kaggle/working/processed/geneformer_tokenize_input/geneformer_input.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.
  tokenised dataset: geneformer_work/tokenized/depmap.dataset

[3/3] Extract embeddings
100%|███████████████████████████████████████████| 72/72 [08:52<00:00,  7.39s/it]
  embeddings: 1140 cell lines x 768 dimensions

DONE
  wrote geneformer_embeddings.csv  (1140 cell lines x 768 dims)

  Next: train_head.py -- train a head on these embeddings and
  compare, on the SAME splits, against the ridge baseline. If the
  embeddings do not beat ridge, that is the finding; report it.


In [22]:
import pandas as pd, glob, os
emb_path = glob.glob("/kaggle/working/**/geneformer_embeddings.csv", recursive=True)[0]
emb = pd.read_csv(emb_path, index_col=0)
print("shape:", emb.shape, "| e.g.", emb.index[:3].tolist())

# copy to the output root so it appears in Kaggle's Output panel to download
emb.to_csv("/kaggle/working/geneformer_embeddings.csv")
print("saved to /kaggle/working/geneformer_embeddings.csv — download it from the Output panel")

shape: (1140, 768) | e.g. ['ACH-000001', 'ACH-000004', 'ACH-000005']
saved to /kaggle/working/geneformer_embeddings.csv — download it from the Output panel
